In [1]:
import os
import syllables
import string
from datasets import load_dataset
from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from transformers import TrainingArguments, set_seed
from mlx_lm import generate
from mlx_tune.chat_templates import get_chat_template, standardize_sharegpt

In [2]:
set_seed(42)

In [3]:
dataset = load_dataset("json", data_files="../../data/processed/haikus_dataset.json", split="train")

In [4]:
dataset

Dataset({
    features: ['conversations'],
    num_rows: 500
})

In [5]:
dataset = standardize_sharegpt(dataset)

In [6]:
max_seq_length = 2048

In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

In [8]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [9]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []
    for convo in convos:
        formatted = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        
        
        texts.append(formatted)
    return {"text": texts}

In [10]:
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Preprocessing complete. Samples formatted: {len(dataset)}")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Preprocessing complete. Samples formatted: 500


In [11]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [12]:
print(f"Training on: {len(train_dataset)} | Evaluating on: {len(eval_dataset)}")

Training on: 450 | Evaluating on: 50


In [13]:
def count_syllables_line(line):
    clean = line.translate(str.maketrans('', '', string.punctuation))
    return sum(syllables.estimate(word) for word in clean.split())

In [14]:
def evaluate_haiku(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) != 3:
        return [0, 0, 0], False
    counts = [count_syllables_line(l) for l in lines]
    is_perfect = (counts == [5, 7, 5])
    return counts, is_perfect

In [15]:
import pandas as pd

def run_internal_eval_detailed(model, tokenizer, test_samples):
    
    results_list = []

    for sample in test_samples:
        user_prompt = sample['messages'][0]['content']

        eval_messages = [
             {"role": "user", "content": user_prompt}
        ]
        
        prompt = tokenizer.apply_chat_template(
            eval_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        response = generate(model, tokenizer, prompt=prompt, max_tokens=64)

        haiku = response.split("<end_of_turn>")[0].strip()

        counts, is_perfect = evaluate_haiku(haiku)
        total_syllables = sum(counts)
        # print(f"Topic: {user_prompt} | Pattern: {counts} | Perfect: {is_perfect}")

        results_list.append({
            "topic": user_prompt,
            "counts": counts,
            "total": total_syllables,
            "is_perfect": is_perfect,
            "error_dist": abs(total_syllables - 17)
        })
        
    return pd.DataFrame(results_list)

In [16]:
# for name, module in model.named_modules():
#     if "proj" in name:
#         print(name)

In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", 
                      "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
)

LoRA configuration set: rank=16, alpha=32, modules=['self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj', 'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'], dropout=0


In [18]:
training_config = SFTConfig(
    output_dir = "mlx_outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    max_steps = 2000,             
    max_length = max_seq_length,
    learning_rate = 2e-4,
    logging_steps = 15,
    # lr_scheduler_type = "constant",
    dataset_text_field = "text",       
)

In [19]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [20]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,      
    train_dataset = train_dataset,
    args = training_config,            
)

Trainer initialized:
  Output dir: mlx_outputs
  Adapter path: mlx_outputs/adapters
  Learning rate: 0.0002
  Iterations: 2000
  Batch size: 2
  LoRA r=16, alpha=32
  Native training: True
  LR scheduler: cosine
  Grad checkpoint: False


In [21]:
print("--- TRAINER INPUT CHECK ---")
print(train_dataset[0]['text'][:100])

--- TRAINER INPUT CHECK ---
<start_of_turn>user
What is the capital of Austria?<end_of_turn>
<start_of_turn>model
Vienna stands 


In [ ]:
print("🍏 Launching Metal-accelerated fine-tuning loop on your M1 GPU...")
trainer.train()
print("✅ Training sequence completely run!")

🍏 Launching Metal-accelerated fine-tuning loop on your M1 GPU...
Starting Fine-Tuning

[Using Native MLX Training]

Applying LoRA adapters...
Applying LoRA to 26 layers: {'rank': 16, 'scale': 2.0, 'dropout': 0, 'keys': ['mlp.down_proj', 'mlp.gate_proj', 'mlp.up_proj', 'self_attn.k_proj', 'self_attn.o_proj', 'self_attn.q_proj', 'self_attn.v_proj']}
✓ LoRA applied successfully to 26 layers
  Trainable LoRA parameters: 364
Preparing training data...
  Detected format: text
✓ Prepared 450 training samples
  Saved to: mlx_outputs/train.jsonl
✓ Created validation set (copied from train)

Training configuration:
  Iterations: 2000
  Batch size: 2
  Learning rate: 0.0002
  LR scheduler: cosine
  Grad checkpoint: False
  Adapter file: mlx_outputs/adapters/adapters.safetensors

Loaded 450 training samples, 450 validation samples
Starting training loop...
Starting training..., iters: 2000


Calculating loss...: 100%|█| 5/5 [00:02<00:00,  

Iter 1: Val loss 11.183, Val took 2.630s


Iter 15: Train loss 3.612, Learning Rate 2.000e-04, It/sec 1.430, Tokens/sec 110.332, Trained Tokens 1157, Peak mem 2.974 GB
Iter 30: Train loss 1.959, Learning Rate 1.999e-04, It/sec 1.977, Tokens/sec 152.085, Trained Tokens 2311, Peak mem 2.975 GB
Iter 45: Train loss 1.789, Learning Rate 1.998e-04, It/sec 1.946, Tokens/sec 150.077, Trained Tokens 3468, Peak mem 2.975 GB
Iter 60: Train loss 1.628, Learning Rate 1.996e-04, It/sec 1.956, Tokens/sec 144.897, Trained Tokens 4579, Peak mem 2.975 GB
Iter 75: Train loss 1.665, Learning Rate 1.993e-04, It/sec 1.952, Tokens/sec 146.032, Trained Tokens 5701, Peak mem 2.975 GB
Iter 90: Train loss 1.662, Learning Rate 1.990e-04, It/sec 1.949, Tokens/sec 147.225, Trained Tokens 6834, Peak mem 2.975 GB
Iter 100: Saved adapter weights to mlx_outputs/adapters/adapters.safetensors and mlx_outputs/adapters/0000100_adapters.safetensors.
Iter 105: Train loss 1.496, Learning Rate 1.987e-04, It/sec 1.962, Tokens/sec 149.633, Trained Tokens 7978, Peak mem 3

Calculating loss...: 100%|█| 5/5 [00:01<00:00,  

...

Iter 600: Val loss 0.349, Val took 1.291s


Iter 600: Train loss 0.374, Learning Rate 1.589e-04, It/sec 1.945, Tokens/sec 146.399, Trained Tokens 45530, Peak mem 3.030 GB
Iter 600: Saved adapter weights to mlx_outputs/adapters/adapters.safetensors and mlx_outputs/adapters/0000600_adapters.safetensors.
Iter 615: Train loss 0.470, Learning Rate 1.570e-04, It/sec 1.955, Tokens/sec 146.899, Trained Tokens 46657, Peak mem 3.030 GB
Iter 630: Train loss 0.364, Learning Rate 1.550e-04, It/sec 1.921, Tokens/sec 144.347, Trained Tokens 47784, Peak mem 3.030 GB
Iter 645: Train loss 0.419, Learning Rate 1.531e-04, It/sec 1.884, Tokens/sec 141.333, Trained Tokens 48909, Peak mem 3.030 GB
Iter 660: Train loss 0.442, Learning Rate 1.510e-04, It/sec 1.864, Tokens/sec 142.001, Trained Tokens 50052, Peak mem 3.030 GB
Iter 675: Train loss 0.367, Learning Rate 1.490e-04, It/sec 1.947, Tokens/sec 147.591, Trained Tokens 51189, Peak mem 3.030 GB
Iter 690: Train loss 0.237, Learning Rate 1.469e-04, It/sec 1.962, Tokens/sec 150.442, Trained Tokens 5233

Calculating loss...: 100%|█| 5/5 [00:01<00:00,  

Iter 800: Val loss 0.226, Val took 1.270s


Iter 800: Saved adapter weights to mlx_outputs/adapters/adapters.safetensors and mlx_outputs/adapters/0000800_adapters.safetensors.
Iter 810: Train loss 0.297, Learning Rate 1.296e-04, It/sec 1.929, Tokens/sec 145.207, Trained Tokens 61426, Peak mem 3.030 GB
Iter 825: Train loss 0.307, Learning Rate 1.273e-04, It/sec 1.939, Tokens/sec 148.272, Trained Tokens 62573, Peak mem 3.030 GB
Iter 840: Train loss 0.266, Learning Rate 1.250e-04, It/sec 1.944, Tokens/sec 152.318, Trained Tokens 63748, Peak mem 3.030 GB
Iter 855: Train loss 0.299, Learning Rate 1.227e-04, It/sec 1.850, Tokens/sec 140.440, Trained Tokens 64887, Peak mem 3.030 GB


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_results(df):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)

    sns.histplot(df['total'], bins=10, color='skyblue', kde=True)
    plt.axvline(17, color='red', linestyle='--', label='Perfect (17)')
    plt.title("Syllable Count Distribution")
    plt.legend()


    plt.subplot(1, 2, 2)
    
    df['is_perfect'].value_index = ["Failure", "Perfect"]
    sns.countplot(x='is_perfect', data=df,  hue='is_perfect', palette='viridis', legend=False)
    plt.title(f"Accuracy: {(df['is_perfect'].mean()*100):.1f}%")

    plt.tight_layout()
    plt.show()

In [ ]:
eval_df = run_internal_eval_detailed(model, tokenizer, eval_dataset)

visualize_results(eval_df)

In [ ]:
eval_df["counts"]

In [ ]:
!pip install mlx-lm

In [ ]:
!python -m mlx_lm fuse --model google/gemma-3-1b-it --adapter-path mlx_outputs/adapters --save-path ../../models/gemmaiku-1b

In [ ]:
from mlx_lm import load, generate

model, tokenizer = load("../../models/gemmaiku-1b")

while True:
    user_input = input("Topic: ")
    prompt = f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"

    response = generate(model, tokenizer, prompt=prompt, max_tokens=64)

    print(response.split("<end_of_turn>")[0].strip())